### Load data and do a quick check

In [ ]:
import json
import pandas as pd
with open("../data/raw/batdongsan.txt", encoding="utf-8") as file:
    df = pd.DataFrame(json.load(file))

In [ ]:
df.head(3)

In [ ]:
df.info()

### Basic cleaning (before merging and more preprocessing)
Rename columns

In [ ]:
rename_map = {
    "Loại hình nhà ở":"property_type",
    "Diện tích đất":"area",
    "Tổng số tầng":"n_floors",
    "Giấy tờ pháp lý":"legal_docs",
    "Số phòng ngủ":"n_bedrooms",
    "Số phòng vệ sinh":"n_bathrooms",
    "Hướng ban công":"balcony_direction",
    "Hướng cửa chính":"facing_direction",
    "Dự án":"project",
    "Tầng số":"floor_num",
    "city":"city_province"
}
df.rename(rename_map, axis=1, inplace=True)
df.head(2)

Extract the numeric values for numeric columns

In [ ]:
import re

def extract_numeric(s:str|None, thousands_sep:bool=True) -> float:
    if not isinstance(s,str) or not s:       # Handle np.nan (a float), or empty strings
        return None                        # Return np.nan cuz None is treated like a value

    results = re.search(r"^(\d+.?\d*,?\d*)\D?", s)
    if not results:
        return None

    num_str = results.group(1)
    if thousands_sep:
        num_str = num_str.replace(".","")
    num_str = num_str.replace(",",".")

    return float(num_str)
    
def extract_measuring_unit(s:str|None) -> str:
    if not isinstance(s,str) or not s:
        return None

    results = re.search(r"^\d+.?\d*,?\d*\s*(\D*)\(*", s)
    if not results:
        return None
    else:
        return results.group(1).replace('(', '').strip()

def extract_dimensions(area_raw:str):
    if not isinstance(area_raw,str) or not area_raw:
        return None

    results = re.search(r"\((.*)x(.*)\)", area_raw)
    if not results:
        return pd.Series((None, None))
    return pd.Series((
        extract_numeric(results.group(1), thousands_sep=False), 
        extract_numeric(results.group(2), thousands_sep=False)
    ))

In [ ]:
df[['dimension_1', 'dimension_2']] = df['area'].apply(extract_dimensions)
df['area_num'] = df['area'].apply(extract_numeric)
df['area_unit'] = df['area'].apply(extract_measuring_unit)
df['n_bedrooms_2'] = df['n_bedrooms'].apply(extract_numeric)
df['n_bathrooms_2'] = df['n_bathrooms'].apply(extract_numeric)
df['price_2'] = pd.to_numeric(df['price'], errors='coerce', downcast='float')
df['n_floors_2'] = pd.to_numeric(df['n_floors'], errors='coerce')

df[['area', 'area_num', 'area_unit', 'dimension_1', 'dimension_2', 
    'n_bedrooms', 'n_bedrooms_2', 'n_bathrooms', 'n_bathrooms_2', 
    'price', 'price_2', 
    'n_floors', 'n_floors_2']].sample(10)

In [ ]:
df[['area', 'area_num', 'dimension_1', 'dimension_2', 
    'n_bedrooms', 'n_bedrooms_2', 'n_bathrooms', 'n_bathrooms_2', 
    'price', 'price_2', 
    'n_floors', 'n_floors_2'
]].info()

In [ ]:
df.loc[df.price_2.isna()]

Observation:
- Non-null counts of `price_2` is lower than the original `price`: "Negotiable" prices 

In [ ]:
df.legal_docs.unique()
# df.loc[df.legal_docs == 'Giấy tờ khác'] = 'Khác'

Extract city and district

In [ ]:
def extract_location_detail(addr: str, level:int) -> str|None:
    '''
    level:int - the level of details to extract from the address. 1 -> Province; 2 -> District
    '''
    if not isinstance(addr, str) or not addr.strip():
        return None
    parts = [p.strip() for p in addr.split(",") if p.strip() != ""]
    if len(parts) >= level:
        return parts[-level]
    return None

In [ ]:
df['city_province'] = df.address.apply(extract_location_detail, level = 1)
df['district'] = df.address.apply(extract_location_detail, level = 2)
df[['address', 'city_province', 'district']]

### Export the file with extracted features

In [ ]:
df.info()

In [ ]:
df_final = df[[
    'property_type', 'price_2', 'area_num', 'area_unit', 'n_bedrooms_2', 'n_bathrooms_2',
    'n_floors_2', 'dimension_2', 'address', 'city_province', 'district','legal_docs', 'facing_direction', 'dimension_1', 'balcony_direction',
    # raw
    'price', 'area', 'n_bedrooms', 'n_bathrooms',
]]
df_final = df_final.rename({
    'price_2': 'price',
    'area_num': 'area',
    'n_bedrooms_2': 'n_bedrooms',
    'n_bathrooms_2': 'n_bathrooms', 
    'dimension_1': 'front_width',
    'n_floors_2': 'n_floors',
    'price': 'raw_price',
    'area': 'raw_area',
    'n_bedrooms': 'raw_n_bedrooms',
    'n_bathrooms': 'raw_n_bathrooms',
}, axis=1)
# dataset source identifier
df_final['scraper'] = 'Thiên'

In [ ]:
df_final.info()

In [ ]:
df_final.to_csv('../data/interim/muaban_net.csv')